# MobileNet — Stage 2: MLP Implementation

This notebook covers the definition, training, and initial evaluation of the Multilayer Perceptron (MLP) model for mobile price range classification.

---

## 1. Imports and Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

sns.set_theme(style='whitegrid')
IMAGES_DIR = 'images/training'
os.makedirs(IMAGES_DIR, exist_ok=True)

In [ ]:
train = pd.read_csv('data/train.csv')

X = train.drop(columns='price_range')
y = train['price_range']

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)

print(f'Train: {X_train.shape} | Val: {X_val.shape}')

---
## 2. MLP Architecture

A Multilayer Perceptron is a feedforward neural network composed of an input layer, one or more hidden layers, and an output layer.

### 2.1 Network structure

```
Input Layer       Hidden Layer 1    Hidden Layer 2    Output Layer
(20 features)  →  (128 neurons)  →  (64 neurons)   →  (4 classes)
```

### 2.2 Forward Pass

At each layer $l$, the output is computed as:

$$a^{(l)} = f\left(W^{(l)} \cdot a^{(l-1)} + b^{(l)}\right)$$

Where:
- $W^{(l)}$ — weight matrix of layer $l$
- $b^{(l)}$ — bias vector of layer $l$
- $f$ — activation function
- $a^{(0)} = x$ — input features

### 2.3 Activation Function — ReLU

Used in hidden layers to introduce non-linearity:

$$\text{ReLU}(x) = \max(0, x)$$

### 2.4 Output Layer — Softmax

Converts the raw scores of the last layer into class probabilities:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

Where $K = 4$ (number of classes) and $\sum_{i=1}^{K} \text{softmax}(z_i) = 1$.

### 2.5 Loss Function — Cross-Entropy

Measures the difference between the predicted probability distribution and the true labels:

$$\mathcal{L} = -\sum_{i=1}^{K} y_i \log(\hat{y}_i)$$

Where $y_i$ is 1 if class $i$ is the true class, and 0 otherwise (one-hot encoding).

### 2.6 Backpropagation — Weight Update

Weights are updated via gradient descent at each iteration:

$$W^{(l)} \leftarrow W^{(l)} - \eta \cdot \frac{\partial \mathcal{L}}{\partial W^{(l)}}$$

Where $\eta$ is the learning rate. The scikit-learn `MLPClassifier` uses the **Adam** optimizer by default, which adapts the learning rate individually for each parameter.

---
## 3. Model Training

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42,
    verbose=False,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

mlp.fit(X_train, y_train)

print(f'Iterations:        {mlp.n_iter_}')
print(f'Train accuracy:    {mlp.score(X_train, y_train):.4f}')
print(f'Val   accuracy:    {mlp.score(X_val, y_val):.4f}')

---
## 4. Learning Curve

The loss curve tracks the value of the cross-entropy loss at each training iteration. A well-trained model shows a monotonically decreasing curve that stabilizes, indicating convergence.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(mlp.loss_curve_, label='Training loss', color='steelblue', linewidth=2)
if mlp.validation_scores_ is not None:
    # validation_scores_ is accuracy; invert for visual comparison with loss
    val_loss = 1 - np.array(mlp.validation_scores_)
    ax.plot(val_loss, label='Validation loss (1 - acc)', color='tomato', linewidth=2, linestyle='--')

ax.set_title('Learning Curve — Cross-Entropy Loss', fontsize=13)
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/01_learning_curve.png', dpi=150)
plt.show()

print(f'Final training loss: {mlp.loss_curve_[-1]:.4f}')

---
## 5. Architecture Summary

| Parameter              | Value                     |
|------------------------|---------------------------|
| Input size             | 20 features               |
| Hidden layer 1         | 128 neurons               |
| Hidden layer 2         | 64 neurons                |
| Output layer           | 4 classes                 |
| Activation (hidden)    | ReLU                      |
| Activation (output)    | Softmax                   |
| Loss function          | Cross-Entropy             |
| Optimizer              | Adam                      |
| Max iterations         | 500                       |
| Early stopping         | Yes (patience = 20)       |
| Random state           | 42                        |

The model is ready for detailed evaluation in Stage 3.